In [1]:
import os
import re
from tqdm import tqdm
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import Counter

from gensim.models import Word2Vec

nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [12]:
import os

DATA_DIR = r"C:\Users\User\PycharmProjects\nlp-course-2025_1\Elen Shahbazyan\Armenian Word2Vec\ilur-news-corpus"

texts = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(".txt"):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                texts.append(f.read())

print(f"Loaded {len(texts)} articles.")


Loaded 12428 articles.


In [13]:
def normalize_armenian(text):
    return text


def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^\w\s՝՞։՛՝՛-]', ' ', text)
    text = normalize_armenian(text)
    return text


def tokenize_text(text):
    sentences = sent_tokenize(text)
    tokenized_sentences = [word_tokenize(clean_text(sent)) for sent in sentences]
    return tokenized_sentences

In [14]:
all_sentences = []
for text in tqdm(texts, desc='Processing texts'):
    all_sentences.extend(tokenize_text(text))

all_sentences = [sent for sent in all_sentences if len(sent) > 0]
print(f'Total sentences: {len(all_sentences)}')

Processing texts: 100%|██████████| 12428/12428 [00:21<00:00, 591.47it/s] 

Total sentences: 26775


In [15]:
word_counts = Counter(word for sent in all_sentences for word in sent)
filtered_sentences = [[word for word in sent if word_counts[word] >= 5] for sent in all_sentences]
filtered_sentences = [sent for sent in filtered_sentences if len(sent) > 0]
print(f'Sentences after removing rare words: {len(filtered_sentences)}')

Sentences after removing rare words: 26702


In [17]:
model = Word2Vec(
    sentences=filtered_sentences,
    vector_size=300,
    window=5,
    sg=1,
    negative=15,
    epochs=20,
    min_count=1
)

In [18]:
model.save('armenian_word2vec.model')
print('Model saved successfully!')

Model saved successfully!


# Test Model


In [21]:
try:
    print("Most similar words to 'հունաստան':")
    print(model.wv.most_similar('հունաստան', topn=10))
except KeyError:
    print("The word 'հունաստան' does not exist in the vocabulary.")

Most similar words to 'հունաստան':
[('շոտլանդիա', 0.8357954621315002), ('իսլանդիա', 0.8231695890426636), ('լիխտենշտեյն', 0.8210620284080505), ('սլովենիա', 0.8113850951194763), ('բազել', 0.8076236248016357), ('մարինո', 0.7846157550811768), ('չեռնոգորիա', 0.7779960036277771), ('մակեդոնիա', 0.7722347378730774), ('սլովակիա', 0.7698950171470642), ('հերցեգովինա', 0.7638715505599976)]
